## Model of "Albireo: Energy-Efficient Acceleration of Convolutional Neural Networks via Silicon Photonics", ISCA 2021

Paper by Kyle Shiflett, Avinash Karanth, Razvan Bunescu, and Ahmed Louri

## Description

Albireo is a silicon photonic accelerator to accelerate convolutional neural networks.
As a photonic accelerator, Albireo leverages the analog-optical (AO) domain in addition
to the analog-electrical (AE) and digital-electrical (DE) domains. The analog-optical
domain has the advantage of low-energy data movement with photonic interconnects.
Additionally, optical components can leverage wavelength division multiplexing (WDM),
where multiple wavelengths of light can be transferred and/or processed at the same time
by the same component.

### Macro Level

Albireo's "macro" is a full accelerator chip. At the chip level, the accelerator
includes a global buffer to store inputs and outputs. An off-chip laser supplies 63
wavelengths of light to the accelerator. 

- *Input Path*: Digital-electrical inputs are fetched from the global buffer. A DAC
  converts them to analog-electrical before a mach-zender-modulator (MZM) and microring
  resonators (MRRs) convert them to optical signals. The analog-optical signals are then
  passed to an analog-optical multicast network. 63 independent wavelengths transmit
  independent inputs, so parallel 63 analog-optical inputs can be sent to the next level
  at once.
- *Weight Path*: Generally, the Albireo paper assumes that weights are pre-loaded at
  levels below this one, so no weight path is shown in the image. If we were to load
  weights (*e.g.*, by placing a DRAM above the accelerator in the hierarchy in our
  model), then weights would be loaded in.
- *Output Path*: Outputs are stored in the global buffer. A digital-electrical network
  transfers outputs from the buffer to lower-level tiles.

Next, there are 9 photonic locally connected groups (PLCGs) in the macro. Inputs are
reused spatially between PLCGs (*i.e.,* each analog-optical interconnect connects to all
PLCGs). Outputs and weights are not reused because each PLCG uses different weights and
generates different outputs.

### Photonic Locally Connected Group (PLCG) Level

The PLCG receives analog-optical inputs, stores digital-electrical weights, and produces
digital-electrical outputs. It includes three Photonic Locally Connected Unit (PLCU)
containers that work together to compute outputs.

- *Input Path*: Analog-optical inputs pass directly to the PLCUs in the PLCG.
- *Weight Path*: A weight cache is used to store weights. It sends digital-electrical
  weights to the PLCUs in the PLCG.
- *Output Path*: The PLCUs generate analog-electrical outputs, which are summed on an
  analog-electrical multicast network. After summation, analog-electrical outputs sent
  to a trans-impedance amplifier (TIA) and ADC to be converted to digital-electrical. An
  output register holds in-progress digital-electrical outputs after they are converted
  by the ADC.
  
Next, there are 3 photonic locally connected units (PLCUs) in each PLCG. Outputs are
reused between PLCUs (*i.e.*, analog-electrical outputs are summed together between
PLCUs in the PLCG). Inputs and weights are not reused between PLCUs.

### Photonic Locally Connected Unit (PLCU) Level

Each PLCU includes an array of optical components that computes MAC operations. The
array is subdivided vertically into three star coupled groups of rows. From the upper
level, a PLCU has three analog-optical input ports that receive 7 analog-optical inputs
each (as different wavelengths). Each of these ports is connected to a different star
coupled group of rows.

- *Input Path*: Analog-optical inputs from the upper level are passed directly to star
  coupled groups of rows.
- *Weight Path*: Digital-electrical weights from the upper level are unicast to the star
  coupled groups of rows.
- *Output Path*: Star coupled groups of rows produce analog-optical outputs that are
  summed on an optical multicast network. The analog-optical outputs are converted to
  analog-electrical outputs by photodiodes before being passed up to the upper level.

Next, there are three star coupled groups of rows in each PLCU. Outputs are reused
between star coupled groups of rows (*i.e.*, analog-optical outputs are summed on
optical interconnects), while inputs and weights are not reused (*i.e.*, each star
coupled group of rows uses a different set of inputs and weights). 

### Star Coupled Group of Rows Level

A star coupled group of rows is a group of three rows in the array.

- *Input Path*: Arrayed waveguide grating (AWG) picks up 7 of the 63 inputs from the
  global 63-parallel-input optical interconnect. These 7 inputs, as parallel
  wavelengths, are sent to a star coupler, which copies the inputs three times. One copy
  of each input is sent to each row in the star coupled group of rows (*i.e.*, each row
  receives all 7 optical-analog inputs). *Note*: The Albireo paper shows the AWGs at the
  PLCG level, but their area calculation suggests that every star coupled group of rows
  has its own AWG, so we model it here.
- *Weight Path*: Each row receives a different electrical-digital weight.
- *Output Path*: The analog-optical outputs from each row are summed on an optical
  multicast network.
  
Next, there are three rows in each star coupled group of rows. Outputs are reused
between rows (*i.e.*, analog-optical outputs are summed on optical interconnects), while
inputs and weights are not reused.

While technically, all rows in a star coupled group of rows are using the same set of
seven inputs, Albireo clever trick to get extra reuse from convolutional sliding
windows. Consider the loop nest below, where we compute 5 outputs (P dimension) using 3
weights (R dimension) and 7 inputs (Q dimension):

``` 

for R in [0..3):
  for P in [0..5):
    Outputs[P] += Inputs[P + R] * Weights[R]
```

In Albireo, each row computes one iteration of the outer R loop. This means that, while
all rows in a star coupled group of rows receive the same 7 inputs, each one actually
only uses 5 of them. Each row computes the same 5 outputs, and these are summed together
spatially between the rows to complete the loop nest. Finally, note that each row in a
star coupled group of rows uses a different weight.

To implement this in Timeloop, we use a `star_coupler` component with coalescing
enabled. Each row reads 5 inputs from the `star_coupler`, and the `star_coupler`
coalesces these overlapping reads into a 7-input read.

### Row Level 
 
A single row computes a single iteration of the outer R loop from the loop nest above.
The row is subdivided into 5 columns.

- *Input Path*: A row uses 5 optical-analog inputs. The inputs pass through a
  mach-zender-modulator (MZM) and are modulated by the weight to produce a
  double-modulated (*i.e.*, input multiplied by weight) signal. Note that the inputs
  arrive as separate wavelengths on the same interconnect, so they can be modulated
  simultaneously by the same weight using one MZM.
- *Weight Path*: A row uses a single digital-electrical weight. The weight is passed
  through a DAC to produce an analog-electrical signal, then used in a MZM to modulate
  (*i.e.*, multiply it with) the inputs.
- *Output Path*: Each column in the row produces a different optical-analog output.
  
In a row, 5 columns all reuse the same weight. Each column uses a different input and
produces a different output. Note a few technical details:

- While multiple columns are connected by the same interconnect, and therefore they may
  technically see the same input, only one column may use a given input. This is because
  each input is encoded on a single wavelength, and the micro-ring resonator (MRR) in
  the column either consume the wavelength or not. For this reason, we enforce that
  inputs may not be reused between columns in a row.
- On the other hand, one weight modules *all* wavelengths that are provided to the row.
  For this reason, the weight can (must be) be reused between columns in a row.
- While the model shows inputs and weights traveling separately to each column, in the
  real architecture each input-weight combination would be combined into a single
  double-modulated optical-analog signal below the MZM. We set up the model such that
  all access counts and reuse patterns are correct despite this simplification.

### Column Level

Finally, each column in the row will compute a single iteration of the inner P loop from
the loop nest above. A MRR is used to select correct double-modulated input-times-weight
wavelength for the column.

- *Input and Weight Paths*: A MRR selects the correct input-and-weight-modulated
  wavelength for the column.
- *Output Path*: Computed outputs are passed directly up to the row.
  
We also include a laser at this level. While the laser is off-chip and not physically
present in a column, we model it here because we want the laser to be powerful enough to
supply all the light needed for all computations in the accelerator. We charge energy
down here so that the laser energy consumption is proportional to the number of
computations performed. With this setup, we can change the number of PLCUs, PLCGs, rows,
and columns and be sure that our laser's energy consumption will scale accordingly.

Below this level, we represent an 8b input x 8b weight -> 8b output MAC as 8x8x8=512
virtualized 1bx1bx1b MAC units.

In [ ]:
from _scripts import *
from tqdm.auto import tqdm

display_important_variables("albireo_isca_2021")
get_spec("albireo_isca_2021").arch

### Area Breakdown

This test replicates the results presented in Fig 9 of the Albireo paper. We show the
area the MRR, MZM, Laser, TIA, DAC, ADC, AWG, and Cache.

Albireo's area is dominated by large optical-analog interconnects: AWGs and star
couplers.

In [ ]:
total_ref_area = 124.6e6
reference_fracs = {
    "AWG": 0.72,
    "Star Coupler": 0.17,
    "Laser": 0.06,
    "MZM": 0.037,
    "MRR": 0.008,
    "ADC": 0.004,
    "Cache": 0.002,
    "Photodiode": 0.002,
    "TIA": 0.001,
    "DAC": 0.0003,
}
component_groups = {
    "AWG": ["ArrayedWaveguideGrating"],
    "Star Coupler": ["StarCoupler"],
    "Laser": ["Laser"],
    "MZM": ["InputMachZehnderModulator", "WeightMachZehnderModulator"],
    "MRR": ["InputMicroRingResonator", "DoubleMRR"],
    "ADC": ["ADC"],
    "Cache": ["WeightCache", "GlobalBuffer"],
    "Photodiode": ["Photodiode"],
    "TIA": ["TIA"],
    "DAC": ["DAC", "WeightDAC"],
}

area = get_area_breakdown("albireo_isca_2021")
area_um2 = {k: v * 1e12 for k, v in area.items()}
modeled = {
    k: sum(area_um2[n] for n in ns if n in area_um2)
    for k, ns in component_groups.items()
}
reference_area = {k: total_ref_area * f for k, f in reference_fracs.items()}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
bar_comparison(
    {"Reference": reference_area, "Modeled": modeled},
    "Component",
    "Area (um²)",
    "Area Breakdown (All Components)",
    ax1,
)
ax1.set_yscale("log")
big = ["AWG", "Star Coupler", "Laser", "MZM"]
bar_comparison(
    {
        "Reference": {k: v for k, v in reference_area.items() if k not in big},
        "Modeled": {k: v for k, v in modeled.items() if k not in big},
    },
    "Component",
    "Area (um²)",
    "Area Breakdown (excluding large components)",
    ax2,
)
ax2.set_yscale("log")
plt.tight_layout()
plt.show()

print(f"{'Group':<15} {'Modeled':>14} {'Reference':>14} {'Ratio':>8}")
for k in component_groups:
    m, r = modeled[k], reference_area[k]
    print(f"{k:<15} {m:11.0f} um² {r:11.0f} um² {m/r:8.2f}x")
print(
    f"Total modeled: {sum(modeled.values()):.0f} um² ({sum(modeled.values()) / 1e6:.2f} mm²)"
)
print(f"Total reference: {total_ref_area:.0f} um² ({total_ref_area / 1e6:.2f} mm²)")

### Energy Breakdown

This test replicates the results presented in Table III of the Albireo paper. We show
the energy the MRR, MZM, Laser, TIA, DAC, ADC, and Cache. Results are shown assuming
three levels of future technology scaling: conservative (high energy), moderate (medium
energy), and aggressive (low energy).

Albireo's energy primarily is consumed by DACs and MRRs. High MRR energy is because
Albireo uses a large array of MRRs. High DAC energy is because Albireo requires many DAC
converts for weights because analog-electrical weights are only reused for 5 MACs each
(low MACs/convert; see the RAELLA paper Titanium law).

In [ ]:
def _run_albireo_energy(scaling):
    spec = get_spec("albireo_isca_2021", add_dummy_main_memory=True)
    spec.variables.scaling = af.LiteralString(scaling)
    spec.variables.cycle_period = 0.2e-9
    return scaling, spec.map_workload_to_arch(print_progress=False)


scalings = ["conservative", "moderate", "aggressive"]
results = dict(
    parallel([delayed(_run_albireo_energy)(s) for s in scalings], pbar="Scaling sweep")
)


def w2j(*args):
    return [y * 0.2e-9 for y in args]


ref_j_per_cycle = {
    "MRR": w2j(7.52, 0.94, 0.38),
    "MZM": w2j(3.45, 0.43, 0.17),
    "Laser": w2j(2.36, 0.09, 0.12 * 5 / 8),
    "TIA": w2j(0.14, 0.07, 0.01),
    "DAC": w2j(7.93, 3.98, 0.80 * 5 / 8),
    "ADC": w2j(1.31, 0.65, 0.13 * 5 / 8),
    "Cache": w2j(0.03, 0.03, 0.03),
}
energy_groups = {
    "MRR": ["DoubleMRR"],
    "MZM": ["WeightMachZehnderModulator", "InputMachZehnderModulator"],
    "Laser": ["Laser"],
    "TIA": ["TIA"],
    "DAC": ["DAC", "WeightDAC"],
    "ADC": ["ADC"],
    "Cache": ["WeightCache", "GlobalBuffer"],
}

fig, axs = plt.subplots(1, len(scalings), figsize=(6 * len(scalings), 5))
print(
    f"{'Scaling':<14} {'Component':<8} {'Modeled (fJ/MAC)':>20} {'Reference (fJ/MAC)':>22} {'Ratio':>8}"
)
print("-" * 80)
for i, s in enumerate(scalings):
    r = results[s]
    n = r.n_computes()
    ec = r.energy(per_component=True)
    modeled_fJ = {
        g: sum(float(ec[c]) for c in cs if c in ec) / n * 1e15
        for g, cs in energy_groups.items()
    }
    ref_fJ = {g: ref_j_per_cycle[g][i] / n * 1e15 for g in energy_groups}
    bar_comparison(
        {"Reference": ref_fJ, "Modeled": modeled_fJ},
        "Component",
        "Energy (fJ/MAC)",
        f"Energy Breakdown - {s} scaling",
        axs[i],
    )
    axs[i].set_yscale("log")
    for g in energy_groups:
        m, ref = modeled_fJ[g], ref_fJ[g]
        print(f"{s:<14} {g:<8} {m:20.2f} {ref:22.2f} {m/ref:8.2f}x")
    print("-" * 80)
plt.tight_layout()
plt.show()

print("Overall TOPS/W:")
for s in scalings:
    pc = results[s].per_compute()
    print(
        f"  {s:<14} TOPS={2/pc.latency()/1e12:8.4f}  TOPS/W={2/pc.energy()/1e12:8.2f}"
    )

### Full-DNN Energy Efficiency and Throughput

This test explores the throughput and energy efficiency of the accelerator for different
DNN layers:

- Albireo has low throughput for the fully-connected layers of all DNNs (last three
  layers of AlexNet, last two layers of VGG16, and last layer of ResNet18). This is
  because Albireo is optimized for convolutional layers and is underutilized when
  running fully-connected layers. This being said, if we look at the total latency and
  energy of each layer, we can see that fully-connected layers are not significant
  contributors to the overall energy and latency for VGG16 and ResNet18, so this
  underutilization has a smaller impact. However, for AlexNet, the fully-connected
  layers are significant contributors to the overall energy and latency, so this
  underutilization has a larger impact.
- VGG16 has consistently high throughput for convolutional layers. This is because VGG16
  uses large weight tensors and convolutional strides of one, which allows Albireo to
  achieve high utilization.
- AlexNet and ResNet18 have lower throughput for convolutional layers. This is due to
  two factors. First, they often use convolutional strides larger than one, for which
  Albireo is not optimized and becomes underutilized. Second, their weight tensor sizes
  vary more than do those of VGG16, and small weight tensors sometimes results in
  underutilization. These effects can be seen especially in ResNet18 layers 7, 12, and
  17, which have one-wide convolutional filters (R=S=1) that result in severe
  underutilization.

We can also see that throughput and energy efficiency are correlated because
underutilization both decreases throughput and increases energy. Energy efficiency
generally varies less than does throughput because underutilization results in fewer
activations of some components, which can somewhat offset the increased energy due to
underutilization.

We note that observing per-layer breakdowns in this way can be a valuable tool to
understand what causes energy and/or throughput differences across full DNNs.

In [ ]:
# The global buffer in Albireo is too small to hold activations, so we upscale it
def _glb_scale_for(dnn):
    if dnn in ("vgg16", "resnet18"):
        return 4
    return 1


def _run_albireo_dnn(dnn):
    workload_path = getattr(af.examples.workloads.compute_in_memory, dnn)
    spec = get_spec("albireo_isca_2021", add_dummy_main_memory=True)
    spec.variables.scaling = af.LiteralString("conservative")
    spec.arch.variables["glb_size_scale"] = _glb_scale_for(dnn)
    spec.workload = af.Workload.from_yaml(
        workload_path, top_key="workload", jinja_parse_data={"BATCH_SIZE": 1}
    )
    spec.renames = af.Renames.from_yaml(
        workload_path, top_key="renames", jinja_parse_data={"BATCH_SIZE": 1}
    )
    spec.mapper.metrics = af.Metrics.ENERGY_DELAY_PRODUCT
    return spec.map_workload_to_arch(print_progress=False)


dnns = ["resnet18", "vgg16", "alexnet"]
dnn_results = {
    d: _run_albireo_dnn(d) for d in tqdm(dnns, desc="Running DNNs on Albireo")
}

fig, axs = plt.subplots(1, 2, figsize=(15, 5))
tops = {d: 2 / dnn_results[d].per_compute().latency() / 1e12 for d in dnns}
tops_w = {d: 2 / dnn_results[d].per_compute().energy() / 1e12 for d in dnns}
bar_comparison({"": tops}, "DNN", "TOPS", "Full-DNN Throughput", axs[0])
bar_comparison({"": tops_w}, "DNN", "TOPS/W", "Full-DNN Energy Efficiency", axs[1])
plt.tight_layout()
plt.show()

print(f"{'DNN':<10} {'TOPS':>10} {'TOPS/W':>10} {'Latency(us)':>14} {'Energy(mJ)':>14}")
for d, r in dnn_results.items():
    pc = r.per_compute()
    print(
        f"{d:<10} {2/pc.latency()/1e12:10.4f} {2/pc.energy()/1e12:10.2f} {r.latency()*1e6:14.2f} {r.energy()*1e3:14.4f}"
    )

for d in dnns:
    fig, axs = plt.subplots(1, 4, figsize=(22, 5))
    for ax, attr, title, ylabel, scale in [
        (axs[0], "tops", "Throughput", "TOPS", 1),
        (axs[1], "tops_per_w", "Energy Efficiency", "TOPS/W", 1),
        (axs[2], "latency", "Latency", "us", 1e6),
        (axs[3], "energy", "Total Energy", "mJ", 1e3),
    ]:
        per_layer = {}
        r = dnn_results[d]
        pc_per = r.per_compute(per_einsum=True)
        if attr == "tops":
            vals = {n: 2 / l / 1e12 for n, l in pc_per.latency(per_einsum=True).items()}
        elif attr == "tops_per_w":
            vals = {n: 2 / e / 1e12 for n, e in pc_per.energy(per_einsum=True).items()}
        elif attr == "latency":
            vals = {n: float(v) * scale for n, v in r.latency(per_einsum=True).items()}
        else:
            vals = {n: float(v) * scale for n, v in r.energy(per_einsum=True).items()}
        for i, (_, v) in enumerate(vals.items()):
            per_layer[i] = v
        bar_comparison(
            {"": {i: per_layer[i] for i in sorted(per_layer)}},
            "Layer",
            f"{title} ({ylabel})",
            f"{d} Per-Layer {title}",
            ax,
        )
    plt.tight_layout()
    plt.show()

### Full DNN Deep Dive

Here we'll do a deep dive into the results for the accelerator running one DNN.

In [ ]:
woarkload_path = af.examples.workloads.compute_in_memory.alexnet
workload = af.Workload.from_yaml(
    woarkload_path, top_key="workload", jinja_parse_data={"BATCH_SIZE": 1}
)
renames = af.Renames.from_yaml(
    woarkload_path, top_key="renames", jinja_parse_data={"BATCH_SIZE": 1}
)

spec = get_spec("albireo_isca_2021", add_dummy_main_memory=True, n_macros=1)

# The Albireo accelerator gloabl buffer is too small to hold activations, so we upscale
# it
spec.arch.variables["glb_size_scale"] = 32

spec.workload = workload
spec.workload.einsums = spec.workload.einsums
spec.renames = renames
spec.mapper.metrics = af.Metrics.ENERGY_DELAY_PRODUCT

result = spec.map_workload_to_arch()
result = result.drop_components_with_zero_energy_and_latency()

In [ ]:
display_dnn_results(result, "Albireo running ResNet18")

Now let's visualize the mapping for the previous result.

In [ ]:
result

### Architectural Exploration

In this test, we explore how architecture-level decisions can affect the energy
efficiency and throughput of the accelerator. One important concept to consider when
designing mixed-signal accelerators is the amount of reuse that the accelerator
leverages in the analog-optical, analog-electrical, digital-optical, digital-electrical
domains. More reuse within a domain permits fewer crossings between domains, which can
reduce the energy and area spent on data converters.

In this test, we vary:

- The number of PLCGs in the accelerator. Analog-optical inputs are reused between
  PLCGs, so increasing the number of PLCGs can decrease input conversion energy.
- The number of PLCUs in the accelerator. Analog-electrical outputs are reused between
  PLCUs, so increasing the number of PLCUs can decrease output conversion energy.
- The MRR topology. Albireo's MRR array can process up to 15 MACs using a convolutional
  sliding window with 7 inputs, 5 weights, and 5 outputs. This topology is often
  underutilized (because it is optimized only for convolutional layers with a stride of
  1), so we try a different structure with 7 inputs, 3 weights, and 7 outputs. This new
  structure does not rely on convolutional sliding windows to be fully utilized, so it
  can achieve consistently-high utilization across many more layer shapes. It also has
  greater analog-optical weight reuse because every analog-optical weight is used for 7
  MACs rather than 5, which permits fewer weight conversions (and therefore less weight
  conversion energy). As a tradeoff, the new structure has less analog-optical input and
  analog-optical output reuse, which can increase the number (and therefore energy) of
  input and output conversion.

We find that increasing the number of PLCGs and PLCUs can increase energy efficiency
through increased reuse. The new MRR topology decreases weight conversion energy and
increases utilization, but increases input and output conversion energy. Overall, it
increases energy efficiency.

The new MRR topology has higher compute density due to the higher utilization. Larger
architectures (more PLCUs and PLCGs) generally have lower compute density because some
layers do not fully utilize them.

In [ ]:
arch_configs = [("Original MRRs", 5, 3, 3), ("New MRRs", 7, 3, 1)]
n_plcu_values = [3, 9, 15]
n_plcg_values = [9, 27, 45]

energy_groups_arch = {
    "Weight Processing": ["WeightMachZehnderModulator", "WeightDAC", "WeightCache"],
    "Input Processing": ["InputMachZehnderModulator", "DAC", "InputMicroRingResonator"],
    "Output Processing": ["ADC", "OutputRegs", "TIA"],
    "Other": ["Laser", "DoubleMRR", "GlobalBuffer"],
}


def _arch_area_mm2(n_columns, n_rows, n_scgr, n_plcu, n_plcg):
    overrides = {
        "n_columns": n_columns,
        "n_rows": n_rows,
        "n_star_coupled_groups_of_rows": n_scgr,
        "n_plcu": n_plcu,
        "n_plcg": n_plcg,
    }
    return (
        sum(
            get_area_breakdown(
                "albireo_isca_2021", variable_overrides=overrides
            ).values()
        )
        * 1e6
    )


def _run_albireo_arch(n_columns, n_rows, n_scgr, n_plcu, n_plcg):
    spec = get_spec("albireo_isca_2021", add_dummy_main_memory=True)
    spec.variables.scaling = af.LiteralString("aggressive")
    spec.arch.variables["n_columns"] = n_columns
    spec.arch.variables["n_rows"] = n_rows
    spec.arch.variables["n_star_coupled_groups_of_rows"] = n_scgr
    spec.arch.variables["n_plcu"] = n_plcu
    spec.arch.variables["n_plcg"] = n_plcg
    spec.workload.rank_sizes["M"] = n_plcg
    spec.workload.rank_sizes["N"] = n_columns
    spec.workload.rank_sizes["K"] = n_plcu * n_scgr * n_rows
    return spec.map_workload_to_arch(print_progress=False)


arch_jobs = [
    (label, ncols, nrows, nscgr, n_plcu, n_plcg)
    for label, ncols, nrows, nscgr in arch_configs
    for n_plcu in n_plcu_values
    for n_plcg in n_plcg_values
]
arch_runs = parallel(
    [
        delayed(_run_albireo_arch)(nc, nr, ns, nu, ng)
        for _, nc, nr, ns, nu, ng in arch_jobs
    ],
    pbar="Architecture sweep",
)

per_arch_summary = {}
for (label, nc, nr, ns, nu, ng), r in zip(arch_jobs, arch_runs):
    nice_label = f"{label}, {nu} PLCUs, {ng} PLCGs"
    area_mm2 = _arch_area_mm2(nc, nr, ns, nu, ng)
    ec = r.energy(per_component=True)
    n = r.n_computes()
    per_g_pj = {
        g: sum(float(ec[c]) for c in cs if c in ec) / n * 1e12
        for g, cs in energy_groups_arch.items()
    }
    pc = r.per_compute()
    per_arch_summary[nice_label] = (
        per_g_pj,
        2 / pc.energy() / 1e12,
        (2 / pc.latency() / 1e12) / area_mm2,
    )

fig, axs = plt.subplots(1, 3, figsize=(22, 6))
bar_stacked(
    {k: v[0] for k, v in per_arch_summary.items()},
    "Architecture",
    "Energy (pJ/MAC)",
    "Per-Architecture Total Energy",
    axs[0],
)
bar(
    {k: v[1] for k, v in per_arch_summary.items()},
    "Architecture",
    "TOPS/W",
    "Per-Architecture Energy Efficiency",
    axs[1],
)
bar(
    {k: v[2] for k, v in per_arch_summary.items()},
    "Architecture",
    "TOPS/mm²",
    "Per-Architecture Compute Density",
    axs[2],
)
plt.tight_layout()
plt.show()

print(f"{'Architecture':<40} {'TOPS/W':>10} {'TOPS/mm^2':>12}")
for k, (_, tw, tmm) in per_arch_summary.items():
    print(f"{k:<40} {tw:10.2f} {tmm:12.6f}")

### Energy Efficiency of Accelerator + Main Memory

This test explores how the accelerator's energy efficiency is affected by
the energy of data fetch from DRAM. It also explores two approaches to
reduce DRAM energy:

- Batching, where weights are fetched once and reused for multiple
  input/output samples. As a tradeoff, this increases latency. We test with
  a batch size of 8.
- Keeping inputs and outputs on-chip in the global buffer between layers
  allows the system to avoid moving them to and from DRAM. As a tradeoff,
  this requires a larger global buffer.

Results are shown assuming three levels of future technology scaling:
conservative (high energy), moderate (medium energy), and aggressive (low
energy).

We find the following:

- For the conservatively-scaled accelerator, DRAM energy is not a
  significant contributor to overall system energy.
- For the aggressively-scaled accelerator, DRAM energy dominates overall
  system energy.
- Batching and keeping inputs and outputs on-chip can each significantly
  reduce DRAM energy.
- To realize the benefits of aggressive scaling, it is critical to
  incorporate both strategies.

In [ ]:
configs = [
    (1, "fetch_all_lpddr4", 1),
    (1, "fetch_weights_lpddr4", 1),
    (8, "fetch_all_lpddr4", 8),
    (8, "fetch_weights_lpddr4", 8),
]
scalings = ["conservative", "moderate", "aggressive"]
energy_groups = {
    "DRAM": ["MainMemory"],
    "Weight Processing": ["WeightMachZehnderModulator", "WeightDAC", "WeightCache"],
    "Input Processing": ["InputMachZehnderModulator", "DAC", "InputMicroRingResonator"],
    "Output Processing": ["ADC", "OutputRegs", "TIA"],
    "Global Buffer": ["GlobalBuffer"],
    "MAC": ["MRR", "DoubleMRR"],
}


def _run_albireo_main_memory(scaling, batch_size, system_setting, glb_scale):
    workload_path = af.examples.workloads.compute_in_memory.resnet18
    parse = {"BATCH_SIZE": batch_size}
    spec = get_spec("albireo_isca_2021", add_dummy_main_memory=False)
    keep = {
        "fetch_all_lpddr4": "All",
        "fetch_weights_lpddr4": "weight",
    }[system_setting]
    spec.arch.nodes.insert(
        0,
        af.arch.Memory(
            name="MainMemory",
            component_class="LPDDR4",
            size=float("inf"),
            tensors={"keep": keep},
            # Nonzero DRAM latency will overwhelm the latency of the accelerator in the
            # following experiment, so while it is unrealistic, just set it to zero so
            # we can focus on energy.
            throughput_scale=float("inf"),
        ),
    )
    spec.variables.scaling = af.LiteralString(scaling)
    spec.arch.variables["glb_size_scale"] = glb_scale
    spec.workload = af.Workload.from_yaml(
        workload_path, top_key="workload", jinja_parse_data=parse
    )
    spec.renames = af.Renames.from_yaml(
        workload_path, top_key="renames", jinja_parse_data=parse
    )
    spec.mapper.metrics = af.Metrics.ENERGY_DELAY_PRODUCT
    return spec.map_workload_to_arch(print_progress=False)


jobs = [(s, b, t, g) for s in scalings for (b, t, g) in configs]
main_memory_runs = {
    (s, b, t): _run_albireo_main_memory(s, b, t, g)
    for s, b, t, g in tqdm(jobs, desc="Main memory sweep")
}

In [ ]:
organized = {}
max_energy_per = {}
for (s, b, t), r in main_memory_runs.items():
    ec = r.per_compute().energy(per_component=True)
    n = int(r.n_computes())
    energy = {
        g: sum(float(ec[c]) for c in cs if c in ec) for g, cs in energy_groups.items()
    }
    onchip = (
        "Inputs & Outputs On-Chip (no DRAM)"
        if "weights" in t
        else "Inputs & Outputs Off-Chip (in DRAM)"
    )
    latency = r.per_compute().latency(per_component=True)
    max_energy_per[s] = max(max_energy_per.get(s, 0), sum(ec.values()))
    organized[(s, b, t)] = {
        "label": f"{onchip}, Batch {b}, {s}",
        "energy": energy,
        "energy_pj_per_mac": float(r.energy()) / n * 1e12,
        "tops_per_w": 2 * n / float(r.energy()) / 1e12,
        "latency": {k: float(v) for k, v in latency.items() if float(v) > 0},
    }

for (s, _, _), data in organized.items():
    data["energy"] = {k: v / max_energy_per[s] for k, v in data["energy"].items()}

fig, axs = plt.subplots(1, 3, figsize=(20, 6))
bar_stacked(
    {organized[k]["label"]: organized[k]["energy"] for k in organized},
    "Architecture",
    "Energy\nNormalized Per Conservative/Moderate/Aggressive",
    "Total Energy",
    axs[0],
)

bar(
    {organized[k]["label"]: organized[k]["tops_per_w"] for k in organized},
    "Architecture",
    "Energy Efficiency (TOPS/W)",
    "Energy Efficiency",
    axs[1],
)

labels = [organized[k]["label"] for k in organized]
x_positions = range(len(labels))
all_components = set()
for k in organized:
    all_components.update(organized[k]["latency"].keys())
all_components = sorted(all_components)
for comp in all_components:
    xs, ys = [], []
    for x, k in zip(x_positions, organized):
        if comp in organized[k]["latency"]:
            xs.append(x)
            ys.append(organized[k]["latency"][comp])
    axs[2].scatter(xs, ys, label=comp, s=80, alpha=0.8)

axs[2].set_xticks(list(x_positions))
axs[2].set_xticklabels(labels, rotation=45, ha="right")
axs[2].set_xlabel("Architecture")
axs[2].set_ylabel("Latency")
axs[2].set_title("Latency by Component")
axs[2].legend()
axs[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print(
    f"{'Scaling':<14} {'Batch':>5} {'System':<22} {'TOPS/W':>10} {'pJ/MAC':>10} {'DRAM pJ/MAC':>13}"
)
for (s, b, t), data in organized.items():
    print(
        f"{s:<14} {b:>5} {t:<22} {data['tops_per_w']:10.2f} {data['energy_pj_per_mac']:10.2f} {data['energy']['DRAM']:13.4f}"
    )